Step 1 downloaded the libraries

pip install sqlalchemy psycopg2-binary

Import libraries

In [1]:
import pandas as pd
from sqlalchemy import create_engine

Ste up the connection with supabse and postgre

In [158]:
host = "db.hbgfqyngiqnoatvvgehz.supabase.co"
database = "postgres"
user = "postgres"
password = "****"
port = "5432"

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
)




In [3]:
df = pd.read_sql(
    "SELECT * FROM merge_data",
    engine
)


1. Understand the data


In [4]:
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (14799, 289)


,opportunity_pk,opportunity_id,badge,careeraddon,opportunity_category,code,cohort,opportunity_created_at,currency_type,current_editor,...,Unnamed: 246,Unnamed: 247,Unnamed: 248,Unnamed: 249,Unnamed: 250,Unnamed: 251,Unnamed: 252,Unnamed: 253,Unnamed: 254,Unnamed: 255
0,Opportunity#,Opportunity#000000000G1RYRMC22H9RYCYSP,"{""sk"":""Badge#000000000G5ABC9367FJ80EQK1"",""ref_...",None,Internship,I152838,"{""sk"":""Cohort#000000000GC4WR598KMWK3G4G4"",""cre...",1.66E+12,USD,None,...,None,None,None,None,None,None,None,None,None,None
1,Opportunity#,Opportunity#0000000010K1EA93ZGXZA0S5NG,"{""sk"":""Badge#00000000107TXSHY7T77DC2GCA"",""ref_...","{""sk"":""CareerAddOn#000000000G3WD2PXWHH5FPQTPH""...",Career,AAL7IGY,"{""sk"":""Cohort#0000000010CZ9TWP5JDCY8R3PN"",""ref...",1.68E+12,USD,None,...,None,None,None,None,None,None,None,None,None,None
2,Opportunity#,Opportunity#0000000010BMP2H74SMG8HWQQ9,"{""sk"":""Badge#0000000010V9WZS8GESDNYADKJ"",""crea...",None,Internship,IRWDLN0,"{""sk"":""Cohort#0000000010N7G42TX81170BMKH"",""cre...",1.77E+12,USD,None,...,None,None,None,None,None,None,None,None,None,None
3,Opportunity#,Opportunity#000000000G1RYRMC22H9RYCYSP,"{""sk"":""Badge#000000000G5ABC9367FJ80EQK1"",""ref_...",None,Internship,I152838,"{""sk"":""Cohort#000000000GC4WR598KMWK3G4G4"",""cre...",1.66E+12,USD,None,...,None,None,None,None,None,None,None,None,None,None
4,Opportunity#,Opportunity#0000000010CTBSXC407YQRNBXZ,"{""sk"":""Badge#0000000010941X2ZRMG2YQ72G5"",""crea...",None,Masterclass,SMVTBAZ,"{""sk"":""Cohort#0000000010CGB9TH0WZVE5TNRJ"",""cre...",1.70E+12,EUR,None,...,None,None,None,None,None,None,None,None,None,None


check data types

In [155]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14799 entries, 0 to 14798
Data columns (total 65 columns):
 #   Column                                               Non-Null Count  Dtype              
---  ------                                               --------------  -----              
 0   opportunity_pk                                       14799 non-null  object             
 1   opportunity_id                                       14799 non-null  object             
 2   badge                                                14398 non-null  object             
 3   careeraddon                                          1887 non-null   object             
 4   opportunity_category                                 14795 non-null  object             
 5   code                                                 14794 non-null  object             
 6   cohort                                               14776 non-null  object             
 7   opportunity_created_at                  

2. checking the missing values

In [5]:
df.isnull().sum().sort_values(ascending=False)

please_identify_your_student_status    14794
Unnamed: 251                           14791
Unnamed: 252                           14791
Unnamed: 253                           14791
Unnamed: 255                           14791
                                       ...  
duration                                   4
opportunity_id                             0
opportunity_pk                             0
pk                                         0
sk                                         0
Length: 289, dtype: int64

missing %

In [6]:
(df.isnull().mean()*100).sort_values(ascending=False)

please_identify_your_student_status    99.966214
Unnamed: 251                           99.945942
Unnamed: 252                           99.945942
Unnamed: 253                           99.945942
Unnamed: 255                           99.945942
                                         ...    
duration                                0.027029
opportunity_id                          0.000000
opportunity_pk                          0.000000
pk                                      0.000000
sk                                      0.000000
Length: 289, dtype: float64

summary table

In [7]:
missing_summary= pd.DataFrame({
    "missing_values": df.isnull().sum(),
    "missing_per":round((df.isnull().sum() / len(df)) * 100, 2),
    'Data Type': df.dtypes
})

missing_summary=missing_summary[
    missing_summary['missing_values']>0
].sort_values(
    by='missing_per',
    ascending=False)

missing_summary.head(100)

,missing_values,missing_per,Data Type
please_identify_your_student_status,14794,99.97,object
Unnamed: 254,14791,99.95,object
Unnamed: 255,14791,99.95,object
Unnamed: 252,14791,99.95,object
Unnamed: 251,14791,99.95,object
...,...,...,...
Unnamed: 165,14760,99.74,object
Unnamed: 166,14760,99.74,object
Unnamed: 164,14760,99.74,object
Unnamed: 162,14759,99.73,object


how  many columns have missing values

In [8]:
df.isna().any().sum()

np.int64(285)

creating missing categories

In [9]:
high_missing= missing_summary[missing_summary['missing_per']>90]

medium_missing= missing_summary[(missing_summary['missing_per']>=40) &
                                 (missing_summary['missing_per']<90)]

low_missing=missing_summary[missing_summary['missing_per']<40]

In [10]:
print("high missing values:", len(high_missing))
print("medium missing values:", len(medium_missing))
print("low missing values:", len(low_missing))

high missing values: 238
medium missing values: 18
low missing values: 29


3. Duplicate values

In [11]:
df.duplicated().sum()

np.int64(0)

4. check unique values

In [12]:
df.nunique().sort_values()

opportunity_pk                                                         1
is_auto_approve                                                        2
payment_status                                                         2
i_will_be_available_from79_pm_ist730930_am_cst_for_meetings_onc        2
i_have_read_and_agreed_to_the_terms_and_conditions                     2
                                                                   ...  
accept_reject_date                                                  6787
work_item_sk                                                        8341
apply_date                                                          9039
created_at                                                         13278
modified_at                                                        13789
Length: 289, dtype: int64

5. constant columns

In [13]:
constant_columns=df.nunique().sort_values()==1
constant_columns

opportunity_pk                                                      True
is_auto_approve                                                    False
payment_status                                                     False
i_will_be_available_from79_pm_ist730930_am_cst_for_meetings_onc    False
i_have_read_and_agreed_to_the_terms_and_conditions                 False
                                                                   ...  
accept_reject_date                                                 False
work_item_sk                                                       False
apply_date                                                         False
created_at                                                         False
modified_at                                                        False
Length: 289, dtype: bool

all named columns

In [14]:
unnamed_columns= [col for col in df.columns if
                  col.startswith("Unnamed")]

print("Total Unnamed columns:", len(unnamed_columns))
unnamed_columns[:20]

Total Unnamed columns: 213


['Unnamed: 43',
 'Unnamed: 44',
 'Unnamed: 45',
 'Unnamed: 46',
 'Unnamed: 47',
 'Unnamed: 48',
 'Unnamed: 49',
 'Unnamed: 50',
 'Unnamed: 51',
 'Unnamed: 52',
 'Unnamed: 53',
 'Unnamed: 54',
 'Unnamed: 55',
 'Unnamed: 56',
 'Unnamed: 57',
 'Unnamed: 58',
 'Unnamed: 59',
 'Unnamed: 60',
 'Unnamed: 61',
 'Unnamed: 62']

check wheter all Unamed columns are highly missing

In [15]:
missing_summary.loc[unnamed_columns].sort_values(
    by='missing_per',
    ascending=False
)

,missing_values,missing_per,Data Type
Unnamed: 251,14791,99.95,object
Unnamed: 255,14791,99.95,object
Unnamed: 253,14791,99.95,object
Unnamed: 254,14791,99.95,object
Unnamed: 252,14791,99.95,object
...,...,...,...
Unnamed: 63,14680,99.20,object
Unnamed: 68,14679,99.19,object
Unnamed: 65,14677,99.18,object
Unnamed: 67,14673,99.15,object


In [16]:
business_high_missing = high_missing[
    ~high_missing.index.str.startswith("Unnamed")
]

business_high_missing

,missing_values,missing_per,Data Type
please_identify_your_student_status,14794,99.97,object
team_creator,14787,99.92,object
i_will_be_available_from79_pm_ist730930_am_cst_for_meetings_onc,14777,99.85,object
from_which_medium_did_you_hear_about_the_internship,14776,99.84,object
why_do_you_want_to_apply_for_this_internship1,14776,99.84,object
can_you_share_an_experience_where_you_proactively_pursued_learn,14751,99.68,object
how_did_you_hear_about_this_opportunity,14749,99.66,object
dropouttransaction,14747,99.65,object
i_understand_that_i_will_have_to_commit_at_least56_hours_each_w,14732,99.55,object
notstartedtransaction,14716,99.44,object


In [17]:
business_high_missing.index.tolist()

['please_identify_your_student_status',
 'team_creator',
 'i_will_be_available_from79_pm_ist730930_am_cst_for_meetings_onc',
 'from_which_medium_did_you_hear_about_the_internship',
 'why_do_you_want_to_apply_for_this_internship1',
 'can_you_share_an_experience_where_you_proactively_pursued_learn',
 'how_did_you_hear_about_this_opportunity',
 'dropouttransaction',
 'i_understand_that_i_will_have_to_commit_at_least56_hours_each_w',
 'notstartedtransaction',
 'i_understand_that_this_is_the_first_step_towards_my_application',
 'transaction_id',
 'relation_id',
 'dropped_out_date',
 'not_started_date',
 'withdraw_date',
 'withdraw_reason',
 'cohort_p2',
 'external_reference_url',
 'team_name',
 'send_for_approval_mechanism_t_c',
 'team_code',
 'reward_awarded_date',
 'completion_date',
 'summary']

drop columns

In [18]:
df.drop(columns=["i_will_be_available_from79_pm_ist730930_am_cst_for_meetings_onc", "i_understand_that_this_is_the_first_step_towards_my_application",
                 'i_understand_that_i_will_have_to_commit_at_least56_hours_each_w','can_you_share_an_experience_where_you_proactively_pursued_learn'
                 ], inplace=True)

## Standarization

checking the columns whether they are same or differ

In [19]:
df[
    [
        "i_agree_to_terms_and_conditions_of_global_shala",
        "i_have_read_and_agreed_to_the_terms_and_conditions"
    ]
].head(50)

,i_agree_to_terms_and_conditions_of_global_shala,i_have_read_and_agreed_to_the_terms_and_conditions
0,None,true
1,None,None
2,None,true
3,None,true
4,None,true
5,None,true
6,true,None
7,None,None
8,true,None
9,None,true


value_counts

In [20]:
# 1
df["i_agree_to_terms_and_conditions_of_global_shala"].value_counts(dropna=False)

# 2
df["i_have_read_and_agreed_to_the_terms_and_conditions"].value_counts(dropna=False)

# 3
(
df["i_agree_to_terms_and_conditions_of_global_shala"]
==
df["i_have_read_and_agreed_to_the_terms_and_conditions"]
).value_counts(dropna=False)

False    14796
True         3
Name: count, dtype: int64

In [21]:
df["i_agree_to_terms_and_conditions_of_global_shala"].value_counts(dropna=False)



i_agree_to_terms_and_conditions_of_global_shala
None     12108
true      2019
TRUE       671
FALSE        1
Name: count, dtype: int64

In [22]:
df["i_have_read_and_agreed_to_the_terms_and_conditions"].value_counts(dropna=False)

i_have_read_and_agreed_to_the_terms_and_conditions
None    8570
true    5690
TRUE     539
Name: count, dtype: int64

Both are same

First standardize them

In [23]:
df["i_agree_to_terms_and_conditions_of_global_shala"] = (
    df["i_agree_to_terms_and_conditions_of_global_shala"]
    .str.lower()
)

In [24]:
df["i_have_read_and_agreed_to_the_terms_and_conditions"] = (
    df["i_have_read_and_agreed_to_the_terms_and_conditions"]
    .str.lower()
)

Create new column

In [25]:
df["terms_conditions_accepted"] = (
    df["i_agree_to_terms_and_conditions_of_global_shala"]
    .combine_first(
        df["i_have_read_and_agreed_to_the_terms_and_conditions"]
    )
)

Valiadte the column

In [26]:
df['terms_conditions_accepted'].value_counts(dropna=False)

terms_conditions_accepted
true     8916
None     5882
false       1
Name: count, dtype: int64

Filling the values with nan

In [27]:
import numpy as np
df['terms_conditions_accepted'].fillna(np.nan)

0        true
1         NaN
2        true
3        true
4        true
         ... 
14794     NaN
14795     NaN
14796     NaN
14797     NaN
14798     NaN
Name: terms_conditions_accepted, Length: 14799, dtype: object

In [28]:
df['terms_conditions_accepted'].value_counts()

terms_conditions_accepted
true     8916
false       1
Name: count, dtype: int64

dropped the columns

In [29]:
df.drop(
    columns=[
        "i_agree_to_terms_and_conditions_of_global_shala",
        "i_have_read_and_agreed_to_the_terms_and_conditions"
    ],
    inplace=True
)

In [30]:
df.columns

Index(['opportunity_pk', 'opportunity_id', 'badge', 'careeraddon',
       'opportunity_category', 'code', 'cohort', 'opportunity_created_at',
       'currency_type', 'current_editor',
       ...
       'Unnamed: 247', 'Unnamed: 248', 'Unnamed: 249', 'Unnamed: 250',
       'Unnamed: 251', 'Unnamed: 252', 'Unnamed: 253', 'Unnamed: 254',
       'Unnamed: 255', 'terms_conditions_accepted'],
      dtype='object', length=284)

checking the value counds and standarization need

In [31]:
df.opportunity_category.value_counts()

opportunity_category
Internship       5180
Competition      2202
Career           2053
Event            1747
Course           1322
Engagement        944
Masterclass       910
JobSimulation     423
Program             7
Xploreu             7
Name: count, dtype: int64

Currency_type

In [32]:
df.currency_type.value_counts()

currency_type
USD     12301
EUR      1121
INR      1002
EURO       31
Name: count, dtype: int64

In [33]:
df['currency_type']= df['currency_type'].replace({'EURO': "EUR"})
df

,opportunity_pk,opportunity_id,badge,careeraddon,opportunity_category,code,cohort,opportunity_created_at,currency_type,current_editor,...,Unnamed: 247,Unnamed: 248,Unnamed: 249,Unnamed: 250,Unnamed: 251,Unnamed: 252,Unnamed: 253,Unnamed: 254,Unnamed: 255,terms_conditions_accepted
0,Opportunity#,Opportunity#000000000G1RYRMC22H9RYCYSP,"{""sk"":""Badge#000000000G5ABC9367FJ80EQK1"",""ref_...",None,Internship,I152838,"{""sk"":""Cohort#000000000GC4WR598KMWK3G4G4"",""cre...",1.66E+12,USD,None,...,None,None,None,None,None,None,None,None,None,true
1,Opportunity#,Opportunity#0000000010K1EA93ZGXZA0S5NG,"{""sk"":""Badge#00000000107TXSHY7T77DC2GCA"",""ref_...","{""sk"":""CareerAddOn#000000000G3WD2PXWHH5FPQTPH""...",Career,AAL7IGY,"{""sk"":""Cohort#0000000010CZ9TWP5JDCY8R3PN"",""ref...",1.68E+12,USD,None,...,None,None,None,None,None,None,None,None,None,None
2,Opportunity#,Opportunity#0000000010BMP2H74SMG8HWQQ9,"{""sk"":""Badge#0000000010V9WZS8GESDNYADKJ"",""crea...",None,Internship,IRWDLN0,"{""sk"":""Cohort#0000000010N7G42TX81170BMKH"",""cre...",1.77E+12,USD,None,...,None,None,None,None,None,None,None,None,None,true
3,Opportunity#,Opportunity#000000000G1RYRMC22H9RYCYSP,"{""sk"":""Badge#000000000G5ABC9367FJ80EQK1"",""ref_...",None,Internship,I152838,"{""sk"":""Cohort#000000000GC4WR598KMWK3G4G4"",""cre...",1.66E+12,USD,None,...,None,None,None,None,None,None,None,None,None,true
4,Opportunity#,Opportunity#0000000010CTBSXC407YQRNBXZ,"{""sk"":""Badge#0000000010941X2ZRMG2YQ72G5"",""crea...",None,Masterclass,SMVTBAZ,"{""sk"":""Cohort#0000000010CGB9TH0WZVE5TNRJ"",""cre...",1.70E+12,EUR,None,...,None,None,None,None,None,None,None,None,None,true
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14794,Opportunity#,Opportunity#0000000010YE3QC6K7XYKEK03P,"{""sk"":""Badge#0000000010T4H7WDDKWCQGVAR2"",""crea...","{""sk"":""CareerAddOn#00000000102BTD5116889EQ7F2""...",Career,AP65EOM,"{""sk"":""Cohort#000000001026J5AKSGNMYZCBNS"",""cre...",1.73E+12,USD,None,...,None,None,None,None,None,None,None,None,None,None
14795,Opportunity#,Opportunity#0000000010YNYQTY10NB9BN3H2,"{""sk"":""Badge#00000000107G1RGKFVCHNKWWZ2"",""crea...",None,Masterclass,SJQIDXS,"{""sk"":""Cohort#0000000010YTVZVPJ0XAZCYGZW"",""cre...",1.76E+12,INR,None,...,None,None,None,None,None,None,None,None,None,None
14796,Opportunity#,Opportunity#0000000010YP965J253KN31Z27,"{""sk"":""Badge#00000000107ZB98474YAJ7X9J1"",""crea...","{""sk"":""CareerAddOn#00000000104RNTP7R4K2V3ZHJ3""...",Career,A49DTHE,"{""sk"":""Cohort#0000000010MPS4RRWJ867GT8V8"",""cre...",1.71E+12,USD,None,...,None,None,None,None,None,None,None,None,None,None
14797,Opportunity#,Opportunity#0000000010YPH375TH13C0SVNZ,"{""sk"":""Badge#0000000010D8RYT42W3S2NVNG9"",""crea...",None,Course,U1S0VA2,"{""sk"":""Cohort#0000000010WFM9QJR97W0DPJ4Q"",""cre...",1.78E+12,USD,None,...,None,None,None,None,None,None,None,None,None,None


In [34]:
df.currency_type.value_counts()

currency_type
USD    12301
EUR     1152
INR     1002
Name: count, dtype: int64

duration_type

In [35]:
df.duration_type.value_counts()

duration_type
weeks       9652
month       1151
hours       1008
days         957
months       679
week         432
minutes      329
years        209
day          168
hour         110
minute        58
year          28
da             8
yearssss       4
Name: count, dtype: int64

In [36]:
df['duration_type']= df['duration_type'].replace({'week':'weeks', 'month':'months', 'hour':'hours', 'day':'days',
                                                  'minute':'minutes','year':'years','yearssss':'years', 'da':'days'
                                                  })

In [37]:
df.duration_type.value_counts()

duration_type
weeks      10084
months      1830
days        1133
hours       1118
minutes      387
years        241
Name: count, dtype: int64

is_archived

In [38]:
df.is_archived.value_counts()

is_archived
TRUE     1303
FALSE     838
Name: count, dtype: int64

In [39]:
df.is_auto_approve.value_counts()

is_auto_approve
FALSE    11504
TRUE      3291
Name: count, dtype: int64

data type

In [40]:
df.last_date_to_apply.dtype

dtype('O')

change the datatype to datetime

In [41]:
df['last_date_to_apply']= pd.to_datetime(pd.to_numeric(df['last_date_to_apply'], errors='coerce'), unit='ms')

In [42]:
df['last_date_to_apply'].head()

0   2023-11-14 22:13:20
1   2023-03-28 10:40:00
2   2026-05-28 20:26:40
3   2023-11-14 22:13:20
4   2023-11-14 22:13:20
Name: last_date_to_apply, dtype: datetime64[ns]

Location

In [43]:
df.location.value_counts()

location
Virtual                          2429
virtual                          1327
WORK FROM HOME                   1296
Work From Home                   1294
wfm                              1197
vitrual                          1185
work from home                   1165
Null                             1157
WFM                              1130
Location                           14
Quia ad ipsum quos                 11
Commodi et aut sunt                10
Opportunity name Page Testing       7
Quis voluptate totam                5
Recusandae Ex volup                 2
Qui voluptatibus nat                2
qwerty                              1
kolkata                             1
Elit cum quis volup                 1
Animi iste aut anim                 1
Aliquam officia est                 1
Name: count, dtype: int64

standardized

In [44]:
df.location.value_counts()

location
Virtual                          2429
virtual                          1327
WORK FROM HOME                   1296
Work From Home                   1294
wfm                              1197
vitrual                          1185
work from home                   1165
Null                             1157
WFM                              1130
Location                           14
Quia ad ipsum quos                 11
Commodi et aut sunt                10
Opportunity name Page Testing       7
Quis voluptate totam                5
Recusandae Ex volup                 2
Qui voluptatibus nat                2
qwerty                              1
kolkata                             1
Elit cum quis volup                 1
Animi iste aut anim                 1
Aliquam officia est                 1
Name: count, dtype: int64

using replace

In [45]:
df['location']= df['location'].replace({'vitrual':'virtual'})

In [46]:
df.location.value_counts()

location
virtual                          2512
Virtual                          2429
WORK FROM HOME                   1296
Work From Home                   1294
wfm                              1197
work from home                   1165
Null                             1157
WFM                              1130
Location                           14
Quia ad ipsum quos                 11
Commodi et aut sunt                10
Opportunity name Page Testing       7
Quis voluptate totam                5
Recusandae Ex volup                 2
Qui voluptatibus nat                2
qwerty                              1
kolkata                             1
Elit cum quis volup                 1
Animi iste aut anim                 1
Aliquam officia est                 1
Name: count, dtype: int64

In [47]:
df['location']= df['location'].replace({'wfm':'Work From Home', 'virtual':'Virtual','work from home':'Work From Home',
                                        'kolkata':'Kolkata'})

In [48]:
import numpy as np
df['location']=df['location'].replace("null", np.nan)

In [49]:
invalid_locations = [
    "location",
    "qwerty",
    "opportunity name page testing",
    "quia ad ipsum quos",
    "commodi et aut sunt",
    "quis voluptate totam",
    "recusandae ex volup",
    "qui voluptatibus nat",
    "elit cum quis volup",
    "animi iste aut anim",
    "aliquam officia est"
]

df["location"] = df["location"].replace(invalid_locations, np.nan)

In [50]:
df.location.value_counts()

location
Virtual                          4941
Work From Home                   3656
WORK FROM HOME                   1296
Null                             1157
WFM                              1130
Location                           14
Quia ad ipsum quos                 11
Commodi et aut sunt                10
Opportunity name Page Testing       7
Quis voluptate totam                5
Recusandae Ex volup                 2
Qui voluptatibus nat                2
Kolkata                             1
Elit cum quis volup                 1
Animi iste aut anim                 1
Aliquam officia est                 1
Name: count, dtype: int64

modified_at

In [51]:
df.modified_at.value_counts().head(20).dtype

dtype('int64')

converting the datatype

In [52]:
df['modified_at']= pd.to_datetime(pd.to_numeric(df['modified_at'], errors='coerce'), unit='ms')

In [53]:
df['modified_at'].head().dtype

dtype('<M8[ns]')

 opportunity name

In [54]:
df['name']= df['name'].str.capitalize()

In [55]:
df['name'].head(50)

0                   Cybersecurity: defensive hack
1                              Eligibility-career
2             Internship automation 1771835946334
3                   Cybersecurity: defensive hack
4                                       Master24`
5                   Cybersecurity: defensive hack
6                 Career automation 1709647541666
7                 Career automation 1726140033066
8                 Career automation 1698989649608
9       Team competition automation 1695294597595
10                Career automation 1741080217135
11                         Teamcompetitionapprove
12               Bulk users applied to internship
13                             Testing comp by ms
14                                 Event reminder
15                Course automation 1715059341844
16            Internship automation 1719123359227
17            Internship automation 1719123359227
18      Team competition automation 1692362229625
19               Bulk users applied to internship


not started date

In [56]:
df.not_started_date.value_counts()

not_started_date
1696086030193      1
1721373111551.0    1
1721373101190.0    1
1709565058624.0    1
1702554119788.0    1
                  ..
1721373092171.0    1
1721373102511.0    1
1727363561006.0    1
1721373090853.0    1
1721373069490.0    1
Name: count, Length: 355, dtype: int64

chnage the datatype

In [57]:
df['not_started_date']= pd.to_datetime(pd.to_numeric(df['not_started_date'], errors='coerce'), unit='ms')

In [58]:
df['not_started_date'].value_counts()

not_started_date
2023-09-30 15:00:30.193    1
2024-07-19 07:11:51.551    1
2024-07-19 07:11:41.190    1
2024-03-04 15:10:58.624    1
2023-12-14 11:41:59.788    1
                          ..
2024-07-19 07:11:32.171    1
2024-07-19 07:11:42.511    1
2024-09-26 15:12:41.006    1
2024-07-19 07:11:30.853    1
2024-07-19 07:11:09.490    1
Name: count, Length: 355, dtype: int64

role

In [59]:
df['role']= df['role'].str.capitalize()

In [60]:
df['role'].head()

0                Testerhsa
1     Quo ab sunt cum elit
2    Testing and debugging
3                Testerhsa
4                     None
Name: role, dtype: object

summary

In [61]:
df['summary'].value_counts()

summary
Summary                                                                                                         471
go                                                                                                               78
a                                                                                                                47
test                                                                                                             47
cc                                                                                                               36
                                                                                                               ... 
dwsd                                                                                                              1
a comprehensive digital growth strategy for Excelerate, by conducting audits and applying the RACE Framework      1
ddd                                                             

nothing useful

In [62]:
df.drop(columns=['summary'],inplace=True)

In [63]:
df.head()

,opportunity_pk,opportunity_id,badge,careeraddon,opportunity_category,code,cohort,opportunity_created_at,currency_type,current_editor,...,Unnamed: 247,Unnamed: 248,Unnamed: 249,Unnamed: 250,Unnamed: 251,Unnamed: 252,Unnamed: 253,Unnamed: 254,Unnamed: 255,terms_conditions_accepted
0,Opportunity#,Opportunity#000000000G1RYRMC22H9RYCYSP,"{""sk"":""Badge#000000000G5ABC9367FJ80EQK1"",""ref_...",None,Internship,I152838,"{""sk"":""Cohort#000000000GC4WR598KMWK3G4G4"",""cre...",1.66E+12,USD,None,...,None,None,None,None,None,None,None,None,None,true
1,Opportunity#,Opportunity#0000000010K1EA93ZGXZA0S5NG,"{""sk"":""Badge#00000000107TXSHY7T77DC2GCA"",""ref_...","{""sk"":""CareerAddOn#000000000G3WD2PXWHH5FPQTPH""...",Career,AAL7IGY,"{""sk"":""Cohort#0000000010CZ9TWP5JDCY8R3PN"",""ref...",1.68E+12,USD,None,...,None,None,None,None,None,None,None,None,None,None
2,Opportunity#,Opportunity#0000000010BMP2H74SMG8HWQQ9,"{""sk"":""Badge#0000000010V9WZS8GESDNYADKJ"",""crea...",None,Internship,IRWDLN0,"{""sk"":""Cohort#0000000010N7G42TX81170BMKH"",""cre...",1.77E+12,USD,None,...,None,None,None,None,None,None,None,None,None,true
3,Opportunity#,Opportunity#000000000G1RYRMC22H9RYCYSP,"{""sk"":""Badge#000000000G5ABC9367FJ80EQK1"",""ref_...",None,Internship,I152838,"{""sk"":""Cohort#000000000GC4WR598KMWK3G4G4"",""cre...",1.66E+12,USD,None,...,None,None,None,None,None,None,None,None,None,true
4,Opportunity#,Opportunity#0000000010CTBSXC407YQRNBXZ,"{""sk"":""Badge#0000000010941X2ZRMG2YQ72G5"",""crea...",None,Masterclass,SMVTBAZ,"{""sk"":""Cohort#0000000010CGB9TH0WZVE5TNRJ"",""cre...",1.70E+12,EUR,None,...,None,None,None,None,None,None,None,None,None,true


tracking ques

In [64]:
df.tracking_questions.value_counts()

tracking_questions
{"code":"Q14YCVL","is_required_for_badge_award":"false","question":"Digital Marketing Assignment","is_frozen":"false","ans_type":"boolean"},{"code":"Q27X70I","is_required_for_badge_award":"false","question":"Additional Notes","is_frozen":"false","ans_type":"string"}                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                119
{"serial_number":"1","code":"QS8HME4","is_required_for_badge_award":"false","question":"Question3","is_frozen":"false","ans_type":"date"},{"code":"Q0MX6GY","question":"Question1","is_frozen":"false","ans_type":"bool

not useful for my analysis

In [65]:
df.drop(columns=["tracking_questions"], inplace=True)

In [66]:
df.head()

,opportunity_pk,opportunity_id,badge,careeraddon,opportunity_category,code,cohort,opportunity_created_at,currency_type,current_editor,...,Unnamed: 247,Unnamed: 248,Unnamed: 249,Unnamed: 250,Unnamed: 251,Unnamed: 252,Unnamed: 253,Unnamed: 254,Unnamed: 255,terms_conditions_accepted
0,Opportunity#,Opportunity#000000000G1RYRMC22H9RYCYSP,"{""sk"":""Badge#000000000G5ABC9367FJ80EQK1"",""ref_...",None,Internship,I152838,"{""sk"":""Cohort#000000000GC4WR598KMWK3G4G4"",""cre...",1.66E+12,USD,None,...,None,None,None,None,None,None,None,None,None,true
1,Opportunity#,Opportunity#0000000010K1EA93ZGXZA0S5NG,"{""sk"":""Badge#00000000107TXSHY7T77DC2GCA"",""ref_...","{""sk"":""CareerAddOn#000000000G3WD2PXWHH5FPQTPH""...",Career,AAL7IGY,"{""sk"":""Cohort#0000000010CZ9TWP5JDCY8R3PN"",""ref...",1.68E+12,USD,None,...,None,None,None,None,None,None,None,None,None,None
2,Opportunity#,Opportunity#0000000010BMP2H74SMG8HWQQ9,"{""sk"":""Badge#0000000010V9WZS8GESDNYADKJ"",""crea...",None,Internship,IRWDLN0,"{""sk"":""Cohort#0000000010N7G42TX81170BMKH"",""cre...",1.77E+12,USD,None,...,None,None,None,None,None,None,None,None,None,true
3,Opportunity#,Opportunity#000000000G1RYRMC22H9RYCYSP,"{""sk"":""Badge#000000000G5ABC9367FJ80EQK1"",""ref_...",None,Internship,I152838,"{""sk"":""Cohort#000000000GC4WR598KMWK3G4G4"",""cre...",1.66E+12,USD,None,...,None,None,None,None,None,None,None,None,None,true
4,Opportunity#,Opportunity#0000000010CTBSXC407YQRNBXZ,"{""sk"":""Badge#0000000010941X2ZRMG2YQ72G5"",""crea...",None,Masterclass,SMVTBAZ,"{""sk"":""Cohort#0000000010CGB9TH0WZVE5TNRJ"",""cre...",1.70E+12,EUR,None,...,None,None,None,None,None,None,None,None,None,true


Learning data columns

1. aceept_reject_date

In [67]:
df.accept_reject_date.value_counts()

accept_reject_date
1683200794088    4
1681990482192    4
1681795910656    4
1683189052109    4
1682071253204    4
                ..
1696503932541    1
1765962926787    1
1768375690403    1
1704962445078    1
1697088776119    1
Name: count, Length: 6787, dtype: int64

change the datatypes

In [68]:
df['accept_reject_date']= pd.to_datetime(pd.to_numeric(df['accept_reject_date'], errors='coerce'), unit='ms')

check the validation

In [69]:
df['accept_reject_date'].head()

0                       NaT
1                       NaT
2   2026-03-26 05:13:45.544
3   2023-04-24 10:53:09.131
4   2023-10-12 05:32:56.119
Name: accept_reject_date, dtype: datetime64[ns]

2. payment_status

In [70]:
df.payment_status.value_counts()

payment_status
unpaid    8529
paid       504
Name: count, dtype: int64

3. modified_at

In [71]:
df['opportunity_modified_at']= pd.to_datetime(pd.to_numeric(df['opportunity_modified_at'], errors='coerce'), unit='ms')

In [72]:
df['opportunity_modified_at'].head()

0   2026-02-02 02:40:00
1   2023-07-22 04:26:40
2   2026-02-02 02:40:00
3   2026-02-02 02:40:00
4   2023-11-14 22:13:20
Name: opportunity_modified_at, dtype: datetime64[ns]

4. created_at

In [73]:
df['opportunity_created_at']= pd.to_datetime(pd.to_numeric(df['opportunity_created_at'], errors='coerce'), unit='ms')

In [74]:
df['opportunity_created_at'].head()

0   2022-08-08 23:06:40
1   2023-03-28 10:40:00
2   2026-02-02 02:40:00
3   2022-08-08 23:06:40
4   2023-11-14 22:13:20
Name: opportunity_created_at, dtype: datetime64[ns]

5. opportunity category

In [75]:
df['opportunity_category'].value_counts()

opportunity_category
Internship       5180
Competition      2202
Career           2053
Event            1747
Course           1322
Engagement        944
Masterclass       910
JobSimulation     423
Program             7
Xploreu             7
Name: count, dtype: int64

6. apply_date

In [76]:
df['apply_date'].dtype

dtype('O')

change the datatype

In [77]:
df['apply_date']= pd.to_datetime(df['apply_date'])

In [78]:
df['apply_date'].head()

0   2023-02-23 08:02:37.290000+00:00
1                                NaT
2   2026-02-23 09:13:02.199000+00:00
3   2023-04-11 08:33:24.750000+00:00
4   2023-10-12 05:32:55.273000+00:00
Name: apply_date, dtype: datetime64[ns, UTC]

In [79]:
df['apply_date'].dtype


datetime64[ns, UTC]

7. from_where_did_you_hear_about_us

In [80]:
df['from_where_did_you_hear_about_us'].value_counts()

from_where_did_you_hear_about_us
Social Media                         3282
Illinois Institute of Technology     2034
GlobalShala Website                   845
Saint Louis University                688
Google Search                         589
social Media                          533
My Friend Or Relative                 309
Illinois Tech                         203
My Friend/Relative                    192
Northeastern University               128
Global Enrichment Program              46
Others                                 38
Other                                  16
Rochester Institute of Technology       8
Hiring Platforms                        1
SLU                                     1
Name: count, dtype: int64

using replace

In [81]:
df['from_where_did_you_hear_about_us']= df['from_where_did_you_hear_about_us'].replace({
    'social Media':'Sociall Media','Others':'Other', 'My Friend Or Relative': 'My Friend/Relative'
})

In [82]:
df.from_where_did_you_hear_about_us.value_counts()

from_where_did_you_hear_about_us
Social Media                         3282
Illinois Institute of Technology     2034
GlobalShala Website                   845
Saint Louis University                688
Google Search                         589
Sociall Media                         533
My Friend/Relative                    501
Illinois Tech                         203
Northeastern University               128
Other                                  54
Global Enrichment Program              46
Rochester Institute of Technology       8
Hiring Platforms                        1
SLU                                     1
Name: count, dtype: int64

8. not_started_date

In [83]:
df['not_started_date'].dtype

dtype('<M8[ns]')

In [84]:
df['not_started_date'].value_counts()

not_started_date
2023-09-30 15:00:30.193    1
2024-07-19 07:11:51.551    1
2024-07-19 07:11:41.190    1
2024-03-04 15:10:58.624    1
2023-12-14 11:41:59.788    1
                          ..
2024-07-19 07:11:32.171    1
2024-07-19 07:11:42.511    1
2024-09-26 15:12:41.006    1
2024-07-19 07:11:30.853    1
2024-07-19 07:11:09.490    1
Name: count, Length: 355, dtype: int64

9. relation_id

In [85]:
df['relation_id'].value_counts()

relation_id
ZYU9E7    2
3HZ8R7    2
WI1TLZ    2
U2SLO0    1
COF5RC    1
         ..
ZB4MLG    1
G5C9YB    1
LLT2VS    1
27PRWA    1
V3Z639    1
Name: count, Length: 241, dtype: int64

10. reward_awarded_date

In [86]:
df['reward_awarded_date'].dtype


dtype('O')

change the data type

In [87]:
df['reward_awarded_date']= pd.to_datetime(pd.to_numeric(df['reward_awarded_date'], errors='coerce'), unit='ms')

In [88]:
df.reward_awarded_date.value_counts()

reward_awarded_date
2024-02-13 13:42:09.453    1
2024-07-19 06:54:11.691    1
2023-02-22 12:51:18.726    1
2025-08-13 06:17:50.510    1
2024-02-13 15:01:28.249    1
                          ..
2023-04-04 08:56:39.068    1
2024-02-07 15:25:10.487    1
2024-02-07 15:25:20.607    1
2026-04-13 10:23:17.401    1
2025-08-13 06:17:57.229    1
Name: count, Length: 1131, dtype: int64

11. completion_date

In [89]:
df['completion_date'].dtype

dtype('O')

change the data type

In [90]:
df['completion_date']= pd.to_datetime(pd.to_numeric(df['completion_date'], errors='coerce'), unit='ms')

In [91]:
df.completion_date.value_counts()

completion_date
2024-02-13 13:42:09.453    1
2024-07-19 06:54:11.691    1
2023-02-22 12:51:18.726    1
2025-08-13 06:17:50.510    1
2024-02-13 15:01:28.249    1
                          ..
2023-04-04 08:56:39.068    1
2024-02-07 15:25:10.487    1
2024-02-07 15:25:20.607    1
2026-04-13 10:23:17.401    1
2025-08-13 06:17:57.229    1
Name: count, Length: 1131, dtype: int64

12. withdraw_reason

In [92]:
df.withdraw_reason.value_counts()

withdraw_reason
Withdraw from opportunity by automation    152
withdraw                                    15
test                                         8
Withdraw                                     7
aa                                           6
                                          ... 
tuyhbnv                                      1
wewrwef                                      1
done by arunava                              1
kfgsadifu                                    1
dcsadad                                      1
Name: count, Length: 177, dtype: int64

not useful

In [93]:
df.drop(columns=['withdraw_reason'], inplace=True)

13. accept_reject_reason

In [94]:
df.accept_reject_reason.value_counts()

accept_reject_reason
Accepted by automation                  1812
This is an auto approved opportunity    1339
Candidate meets all requirements         738
Strong application                       338
accept                                   278
                                        ... 
jkjn                                       1
asa                                        1
dfds                                       1
sdfghjkl                                   1
not                                        1
Name: count, Length: 837, dtype: int64

not mush useful but keeping

14. Team name

In [95]:
df.team_name.value_counts()

team_name
test                          37
MSD                           16
abc                           14
Test                          11
team                           9
                              ..
globalshala                    1
Team Compete 1726066535016     1
Xy                             1
HI                             1
Team Compete 1696418086931     1
Name: count, Length: 526, dtype: int64

In [96]:
df["team_name"] = (
    df["team_name"]
    .str.strip()
    .str.lower()
)

In [97]:
df["team_name"].value_counts().head(20)

team_name
test             50
msd              16
abc              14
fcb              13
team             11
testing           9
te                8
aaa               8
cs bulls          6
101 compete       6
test team         6
123456            6
team1             6
qwerty            6
test1             6
tea               5
excelerate        5
qwertyu           5
team_karthick     5
abcd              5
Name: count, dtype: int64

In [98]:
df[df["team_name"].isin(["test","abc","team","hi","xy"])]

,opportunity_pk,opportunity_id,badge,careeraddon,opportunity_category,code,cohort,opportunity_created_at,currency_type,current_editor,...,Unnamed: 247,Unnamed: 248,Unnamed: 249,Unnamed: 250,Unnamed: 251,Unnamed: 252,Unnamed: 253,Unnamed: 254,Unnamed: 255,terms_conditions_accepted
9,Opportunity#,Opportunity#0000000010J6Z7APJ5SW73XVN8,"{""sk"":""Badge#0000000010Z6S87GX2GRFM7CVX"",""crea...",None,Competition,M8VD087,"{""sk"":""Cohort#00000000109GB03XBGV1N1TD2R"",""cre...",2023-11-14 22:13:20,USD,None,...,None,None,None,None,None,None,None,None,None,true
461,Opportunity#,Opportunity#0000000010GDGG75M44J75GQH1,"{""sk"":""Badge#0000000010F00MY947V6C87VBZ"",""crea...",None,Competition,MQWM1QC,"{""sk"":""Cohort#0000000010PCP5HAB7S1RNR9TY"",""cre...",2023-11-14 22:13:20,EUR,None,...,None,None,None,None,None,None,None,None,None,true
468,Opportunity#,Opportunity#000000001095TD3ZZX538NHV5W,"{""sk"":""Badge#0000000010HPWVK0SHF89KTCRZ"",""crea...",None,Competition,MISHDKK,"{""sk"":""Cohort#0000000010Z889RTXW2RS9070Q"",""cre...",2023-11-14 22:13:20,EUR,None,...,None,None,None,None,None,None,None,None,None,true
609,Opportunity#,Opportunity#0000000010AVK8K8XG1W4R1YZ0,"{""sk"":""Badge#0000000010Y0CQSPW35R279X77"",""crea...",None,Competition,MDEORB8,"{""sk"":""Cohort#00000000108AKM8APNRTXN6YDM"",""cre...",2024-03-09 16:00:00,INR,None,...,None,None,None,None,None,None,None,None,None,true
660,Opportunity#,Opportunity#0000000010W2GPM8EVJ0KA2JT6,"{""sk"":""Badge#0000000010JZ3SGM21VGV4DSPF"",""ref_...",None,Competition,MY76VLQ,"{""sk"":""Cohort#0000000010WP6XME4KVJRN5X2T"",""ref...",2023-07-22 04:26:40,USD,None,...,None,None,None,None,None,None,None,None,None,true
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14328,Opportunity#,Opportunity#0000000010BWBVJ6Z8G6R7E2ZK,"{""sk"":""Badge#0000000010Q3EP6MKRAC5SDZGX"",""crea...",None,Competition,MNEBV5Q,"{""sk"":""Cohort#0000000010KZD5QZ7ZTAGHG0VB"",""cre...",2023-07-22 04:26:40,USD,None,...,None,None,None,None,None,None,None,None,None,true
14356,Opportunity#,Opportunity#0000000010D5FWQYDPMWKYZFTD,"{""sk"":""Badge#0000000010JAA1BYJWKSXCTNBJ"",""crea...","{""sk"":""CareerAddOn#00000000101K1ZZ91CRHR4SYAM""...",Competition,MBUIENU,"{""sk"":""Cohort#0000000010RWZCK8SR7GB13S34"",""cre...",2025-10-09 08:53:20,USD,None,...,None,None,None,None,None,None,None,None,None,true
14357,Opportunity#,Opportunity#0000000010D5FWQYDPMWKYZFTD,"{""sk"":""Badge#0000000010JAA1BYJWKSXCTNBJ"",""crea...","{""sk"":""CareerAddOn#00000000101K1ZZ91CRHR4SYAM""...",Competition,MBUIENU,"{""sk"":""Cohort#0000000010RWZCK8SR7GB13S34"",""cre...",2025-10-09 08:53:20,USD,None,...,None,None,None,None,None,None,None,None,None,true
14575,Opportunity#,Opportunity#0000000010C5C2A35BS0GNP5K4,"{""sk"":""Badge#00000000107DB21Z21AZR507SW"",""crea...",None,Competition,MTTJLE2,"{""sk"":""Cohort#000000001034AS0QQFCPW8VDD1"",""cre...",2026-02-02 02:40:00,USD,None,...,None,None,None,None,None,None,None,None,None,true


In [99]:
import numpy as np

invalid_team_names = [
    "test",
    "abc",
    "team",
    "hi",
    "xy",
    "qwerty",
    "aaa",
    "te",
    "ggg"
]

df["team_name"] = (
    df["team_name"]
    .str.strip()
    .str.lower()
)

df["team_name"] = df["team_name"].replace(invalid_team_names, np.nan)

In [100]:
df.team_name.value_counts()

team_name
msd                           16
fcb                           13
testing                        9
cs bulls                       6
101 compete                    6
                              ..
team compete 1696410773715     1
qw                             1
team compete 1696406810584     1
team compete 1696410568988     1
team compete 1696418086931     1
Name: count, Length: 498, dtype: int64

5. team_creator

In [101]:
df.team_creator.value_counts()

team_creator
akshith.pulluri28@gmail.com    4
akshith@globalshala.com        2
shringinituk@gmail.com         2
mshringi22@gmail.com           1
support@vempower.org           1
shivampandeyy50@gmail.com      1
manish.shringi.58@gmail.com    1
Name: count, dtype: int64

16. send_for_approval_mechanism_t_c

In [102]:
df.send_for_approval_mechanism_t_c.value_counts()

send_for_approval_mechanism_t_c
automatic    645
manual       403
Name: count, dtype: int64

17. please_identify_your_student_status

In [103]:
df.please_identify_your_student_status.value_counts()

please_identify_your_student_status
prospectiveUndergraduateStudent    4
prospectiveGraduateStudents        1
Name: count, dtype: int64

18. why_do_you_want_to_apply_for_this_internship1

In [104]:
df.why_do_you_want_to_apply_for_this_internship1.value_counts()

why_do_you_want_to_apply_for_this_internship1
Will you be available from 7-9pm IST (7:30-9:30 am CST) for meetings once a week.                                                                                                                                                                            5
Why do you want to apply for this internship? * Why do you want to apply for this internship? * Why do you want to apply for this internship? * Why do                                                                                                       2
Why do you want to apply for this internship                                                                                                                                                                                                                 2
Will you be available from 7-9pm IST (7:30-9:30 am CST) for meetings once a week.%0AWill you be available from 7-9pm IST (7:30-9:30 am CST) for mgdfgdfg                                     

drop the column

In [105]:
df.drop(columns=['why_do_you_want_to_apply_for_this_internship1'], inplace=True)

19. from_which_medium_did_you_hear_about_the_internship

In [106]:
df.from_which_medium_did_you_hear_about_the_internship.value_counts()

from_which_medium_did_you_hear_about_the_internship
workshopFairOrganisedBySaintLouisUniversity    8
printedLetter                                  8
socialMedia                                    4
recommendedByAFriendFamilyCounsellor           2
googleSearch                                   1
Name: count, dtype: int64

standarize 

In [107]:
df["from_which_medium_did_you_hear_about_the_internship"] = (
    df["from_which_medium_did_you_hear_about_the_internship"]
    .replace({
        "workshopFairOrganisedBySaintLouisUniversity": "Workshop / Fair (Saint Louis University)",
        "printedLetter": "Printed Letter",
        "socialMedia": "Social Media",
        "recommendedByAFriendFamilyCounsellor": "Friend / Family / Counsellor",
        "googleSearch": "Google Search"
    })
)

In [108]:
df.from_which_medium_did_you_hear_about_the_internship.value_counts()

from_which_medium_did_you_hear_about_the_internship
Workshop / Fair (Saint Louis University)    8
Printed Letter                              8
Social Media                                4
Friend / Family / Counsellor                2
Google Search                               1
Name: count, dtype: int64

21. why_do_you_want_to_apply_for_this_internship

In [109]:
df.why_do_you_want_to_apply_for_this_internship.value_counts()

why_do_you_want_to_apply_for_this_internship
Applying from cypress automation testing.          1006
Testing                                             217
Learning                                            179
Why do you want to apply for this Internship? *     171
12345678                                            155
                                                   ... 
Opportunity name five day reminder                    1
ewerrwr                                               1
SApply                                                1
m                                                     1
nv.sindhu94@gmail.com                                 1
Name: count, Length: 500, dtype: int64

In [110]:
df.drop(columns=['why_do_you_want_to_apply_for_this_internship'], inplace=True)

22. withdraw_date

In [111]:
df.withdraw_date.dtype

dtype('O')

change the datatype

In [112]:
df['withdraw_date']= pd.to_datetime(pd.to_numeric(df['withdraw_date'], errors='coerce'), unit='ms')

In [113]:
df.withdraw_date.value_counts()

withdraw_date
2026-02-26 10:55:05.951    1
2024-04-07 17:11:06.955    1
2023-04-04 11:53:00.271    1
2023-08-09 06:37:53.292    1
2023-06-01 12:21:27.052    1
                          ..
2023-04-21 07:46:45.225    1
2023-02-28 05:44:14.061    1
2026-02-27 09:01:43.956    1
2023-05-12 09:05:18.351    1
2023-08-29 18:44:03.114    1
Name: count, Length: 385, dtype: int64

23. created_at

In [114]:
df.created_at.dtype

dtype('O')

In [115]:
df['created_at']= pd.to_datetime(pd.to_numeric(df['created_at'], errors='coerce'), unit='ms')

In [116]:
df.created_at.value_counts()

created_at
2024-12-31 00:00:00.000    5
2024-08-13 16:13:35.806    4
2023-10-24 10:36:38.814    4
2024-08-14 06:26:09.616    4
2024-08-14 07:00:48.279    4
                          ..
2025-09-09 04:17:33.332    1
2026-03-02 06:28:14.557    1
2023-12-01 09:59:40.153    1
2024-10-22 11:10:19.936    1
2024-04-03 07:23:20.303    1
Name: count, Length: 13250, dtype: int64

24. external_reference_url

In [117]:
df.external_reference_url.value_counts()

external_reference_url
https://sites.google.com/                                                                                                    197
https://www.google.com/                                                                                                      131
http://localhost:4200/user-profile/personal-information                                                                       75
http://localhost:4200/                                                                                                        37
http://localhost:4200/opportunities/Global%20Internships?category=Internship&isFilterByEligibility=0                          25
                                                                                                                            ... 
http://localhost:63652/user-profile/personal-information                                                                       1
http://localhost:4200/list/assignment-review?opportunitySk=00000000109393E

25. i_understand_that_this_is_the_first_step_towards_my_application

In [118]:
df.how_did_you_hear_about_this_opportunity.value_counts()

how_did_you_hear_about_this_opportunity
socialMedia                                18
emailFromRochesterInstituteofTechnology     8
emailFromSaintLouisUniversity               7
googleSearch                                6
Gwendolyn Brooks College Prep               2
Lindblom Math and Science Academy           2
myFriendRelative                            1
'"size'":'"25566'"                          1
other                                       1
Kruu Website                                1
WorkItem#0000000010E55KPBJPHMBGDTGW         1
Saint Louis University                      1
Email from Saint Louis University           1
Name: count, dtype: int64

In [119]:
import numpy as np

df["how_did_you_hear_about_this_opportunity"] = (
    df["how_did_you_hear_about_this_opportunity"]
    .replace({
        "socialMedia": "Social Media",
        "googleSearch": "Google Search",
        "emailFromRochesterInstituteofTechnology": "Email - Rochester Institute of Technology",
        "emailFromSaintLouisUniversity": "Email - Saint Louis University",
        "Email from Saint Louis University": "Email - Saint Louis University",
        "myFriendRelative": "Friend / Relative",
        "Saint Louis University": "Saint Louis University",
        "Gwendolyn Brooks College Prep": "Gwendolyn Brooks College Prep",
        "Lindblom Math and Science Academy": "Lindblom Math and Science Academy",
        "Kruu Website": "Kruu Website",
        "other": "Other",
        
    
        
    })
)

In [120]:
df.loc[
    df["how_did_you_hear_about_this_opportunity"].str.contains("size", na=False),
    "how_did_you_hear_about_this_opportunity"
] = np.nan

In [121]:
df.how_did_you_hear_about_this_opportunity.value_counts()

how_did_you_hear_about_this_opportunity
Social Media                                 18
Email - Saint Louis University                8
Email - Rochester Institute of Technology     8
Google Search                                 6
Gwendolyn Brooks College Prep                 2
Lindblom Math and Science Academy             2
Friend / Relative                             1
Other                                         1
Kruu Website                                  1
WorkItem#0000000010E55KPBJPHMBGDTGW           1
Saint Louis University                        1
Name: count, dtype: int64

26. dropped_out_date

In [122]:
df.dropped_out_date.value_counts()

dropped_out_date
1692955720353.0    2
1723191525702.0    1
1693374251625.0    1
1691390655436.0    1
1687507388506.0    1
                  ..
1705938799712      1
1690809330803      1
1705938794448      1
1727358014605      1
1702556876717      1
Name: count, Length: 320, dtype: int64

change the datatype

In [123]:
df['dropped_out_date']= pd.to_datetime(pd.to_numeric(df['dropped_out_date'], errors='coerce'), unit='ms')

In [124]:
df['dropped_out_date'].value_counts()

dropped_out_date
2023-08-25 09:28:40.353    2
2024-01-24 12:07:13.948    1
2024-01-22 15:53:18.470    1
2023-11-03 10:40:49.057    1
2025-03-31 13:13:02.619    1
                          ..
2023-06-21 13:22:17.607    1
2024-01-23 08:47:56.617    1
2024-01-23 08:47:58.934    1
2024-03-01 10:38:55.790    1
2024-03-01 13:31:05.574    1
Name: count, Length: 319, dtype: int64

27. appear_to_wishlist

In [125]:
df.appear_in_wishlist.value_counts()

appear_in_wishlist
'"is_frozen'":'"false'"                                      865
{'"serial_number'":'"1'"                                     458
{'"serial_number'":'"0'"                                     403
'"question'":'"Additional Notes'"                            392
'"ans_type'":'"boolean'"}                                    159
                                                            ... 
{'"code'":'"QQG3RE4'"                                          1
'"ref_properties'":'"{\'"relation_id\'":\'"A5JRYZ\'"}'"}"      1
{'"code'":'"QDADXNF'"                                          1
{'"code'":'"QD7BQZH'"                                          1
{'"code'":'"QDA418A'"                                          1
Name: count, Length: 1159, dtype: int64

In [126]:
df.drop(columns=["appear_in_wishlist"], inplace=True)

In [127]:
df

,opportunity_pk,opportunity_id,badge,careeraddon,opportunity_category,code,cohort,opportunity_created_at,currency_type,current_editor,...,Unnamed: 247,Unnamed: 248,Unnamed: 249,Unnamed: 250,Unnamed: 251,Unnamed: 252,Unnamed: 253,Unnamed: 254,Unnamed: 255,terms_conditions_accepted
0,Opportunity#,Opportunity#000000000G1RYRMC22H9RYCYSP,"{""sk"":""Badge#000000000G5ABC9367FJ80EQK1"",""ref_...",None,Internship,I152838,"{""sk"":""Cohort#000000000GC4WR598KMWK3G4G4"",""cre...",2022-08-08 23:06:40,USD,None,...,None,None,None,None,None,None,None,None,None,true
1,Opportunity#,Opportunity#0000000010K1EA93ZGXZA0S5NG,"{""sk"":""Badge#00000000107TXSHY7T77DC2GCA"",""ref_...","{""sk"":""CareerAddOn#000000000G3WD2PXWHH5FPQTPH""...",Career,AAL7IGY,"{""sk"":""Cohort#0000000010CZ9TWP5JDCY8R3PN"",""ref...",2023-03-28 10:40:00,USD,None,...,None,None,None,None,None,None,None,None,None,None
2,Opportunity#,Opportunity#0000000010BMP2H74SMG8HWQQ9,"{""sk"":""Badge#0000000010V9WZS8GESDNYADKJ"",""crea...",None,Internship,IRWDLN0,"{""sk"":""Cohort#0000000010N7G42TX81170BMKH"",""cre...",2026-02-02 02:40:00,USD,None,...,None,None,None,None,None,None,None,None,None,true
3,Opportunity#,Opportunity#000000000G1RYRMC22H9RYCYSP,"{""sk"":""Badge#000000000G5ABC9367FJ80EQK1"",""ref_...",None,Internship,I152838,"{""sk"":""Cohort#000000000GC4WR598KMWK3G4G4"",""cre...",2022-08-08 23:06:40,USD,None,...,None,None,None,None,None,None,None,None,None,true
4,Opportunity#,Opportunity#0000000010CTBSXC407YQRNBXZ,"{""sk"":""Badge#0000000010941X2ZRMG2YQ72G5"",""crea...",None,Masterclass,SMVTBAZ,"{""sk"":""Cohort#0000000010CGB9TH0WZVE5TNRJ"",""cre...",2023-11-14 22:13:20,EUR,None,...,None,None,None,None,None,None,None,None,None,true
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14794,Opportunity#,Opportunity#0000000010YE3QC6K7XYKEK03P,"{""sk"":""Badge#0000000010T4H7WDDKWCQGVAR2"",""crea...","{""sk"":""CareerAddOn#00000000102BTD5116889EQ7F2""...",Career,AP65EOM,"{""sk"":""Cohort#000000001026J5AKSGNMYZCBNS"",""cre...",2024-10-27 03:33:20,USD,None,...,None,None,None,None,None,None,None,None,None,None
14795,Opportunity#,Opportunity#0000000010YNYQTY10NB9BN3H2,"{""sk"":""Badge#00000000107G1RGKFVCHNKWWZ2"",""crea...",None,Masterclass,SJQIDXS,"{""sk"":""Cohort#0000000010YTVZVPJ0XAZCYGZW"",""cre...",2025-10-09 08:53:20,INR,None,...,None,None,None,None,None,None,None,None,None,None
14796,Opportunity#,Opportunity#0000000010YP965J253KN31Z27,"{""sk"":""Badge#00000000107ZB98474YAJ7X9J1"",""crea...","{""sk"":""CareerAddOn#00000000104RNTP7R4K2V3ZHJ3""...",Career,A49DTHE,"{""sk"":""Cohort#0000000010MPS4RRWJ867GT8V8"",""cre...",2024-03-09 16:00:00,USD,None,...,None,None,None,None,None,None,None,None,None,None
14797,Opportunity#,Opportunity#0000000010YPH375TH13C0SVNZ,"{""sk"":""Badge#0000000010D8RYT42W3S2NVNG9"",""crea...",None,Course,U1S0VA2,"{""sk"":""Cohort#0000000010WFM9QJR97W0DPJ4Q"",""cre...",2026-05-28 20:26:40,USD,None,...,None,None,None,None,None,None,None,None,None,None


## Removing Unnamed columns all 

unnamed 43

In [128]:
df['Unnamed: 100'].value_counts()

Unnamed: 100
{'"rs'":'"false'"                                4
{'rs'":'"false'"                                 2
'"ra'":'"false'"}                                2
\'"mt\'":\'"1766054029464\'"}'"                  1
\'"ct\'":\'"1766091068644\'"                     1
                                                ..
\'"ca\'":\'"1760094759244\'"}'"]                 1
\'"mt\'":\'"1766100767803\'"}'"                  1
{'"Page'":['"{\'"sk\'":\'"Pe#1761553205223\'"    1
\'"mt\'":\'"1770834814233\'"}'"]                 1
\'"ct\'":\'"1766149020844\'"                     1
Name: count, Length: 97, dtype: int64

REmove all the columns names started with unnamed

In [129]:
df= df.loc[:, ~df.columns.str.startswith("Unnamed")]

verifying

In [130]:
print(df.filter(regex="^Unnamed").columns)

Index([], dtype='object')


shape of the dataset

In [131]:
print(df.shape)

(14799, 65)


REcalculate missing values

In [132]:
missing_summary = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Missing Percentage': round(df.isnull().mean()*100,2)
})

missing_summary = missing_summary[
    missing_summary["Missing Values"] > 0
].sort_values(
    by="Missing Percentage",
    ascending=False
)

missing_summary

,Missing Values,Missing Percentage
please_identify_your_student_status,14794,99.97
team_creator,14787,99.92
from_which_medium_did_you_hear_about_the_internship,14776,99.84
how_did_you_hear_about_this_opportunity,14750,99.67
dropouttransaction,14747,99.65
...,...,...
opportunity_modified_at,4,0.03
last_date_to_apply,5,0.03
is_auto_approve,4,0.03
duration,4,0.03


## Handling missing values

1. event_columns = [
    "accept_reject_date",
    "accept_reject_reason",
    "completion_date",
    "reward_awarded_date",
    "withdraw_date",
    "withdraw_reason",
    "dropped_out_date",
    "not_started_date",
    "transaction_id",
    "dropouttransaction",
    "notstartedtransaction",
    "relation_id",
    "team_name",
    "team_code",
    "cohort_p2",
    "external_reference_url",
    "application_id"
]

keeping all the columns as it is 

2. Fill Text Columns

In [133]:
text_columns = [
    "summary",
    "short_description",
    "long_description",
    "role_responsibility",
    "testimonial"
]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].fillna("Not Available")

C:\Users\DELL\AppData\Local\Temp\ipykernel_2656\1223458342.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].fillna("Not Available")


3. Fill Categorical Columns

In [134]:
categorical_columns = [
    "location",
    "category",
    "status",
    "assigned_cohort",
    "currency_type",
    "duration_type",
    "payment_status"
]

for col in categorical_columns:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

C:\Users\DELL\AppData\Local\Temp\ipykernel_2656\3612923521.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].fillna("Unknown")


4. Fill Boolean Columns

In [135]:
boolean_columns = [
    "is_archived",
    "is_auto_approve",
    "microscholarship"
]

for col in boolean_columns:
    if col in df.columns:
        df[col] = df[col].fillna(False)

C:\Users\DELL\AppData\Local\Temp\ipykernel_2656\3959815340.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].fillna(False)


5. Numeric Columns

In [136]:
df["fee"].describe()

count     5749
unique     138
top        0.0
freq      3600
Name: fee, dtype: object

In [137]:
df.fee.dtype

dtype('O')

In [138]:
df.fee.isnull().sum()

np.int64(9050)

In [139]:
df["fee"].value_counts(dropna=False).head(20)

fee
None     9050
0.0      3600
0         939
1.0       537
1         144
100.0     124
10.0       81
100        34
10         24
2.0        19
50.0       13
5.0         8
20.0        8
23.0        6
49.0        5
200.0       5
111.0       5
12.0        5
33.0        4
78.0        4
Name: count, dtype: int64

Change the data type

In [140]:
df["fee"] = pd.to_numeric(df["fee"], errors="coerce")

C:\Users\DELL\AppData\Local\Temp\ipykernel_2656\2787056420.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["fee"] = pd.to_numeric(df["fee"], errors="coerce")


In [141]:
df["fee"] = df["fee"].fillna(df["fee"].median())
df["reward"] = df["reward"].fillna(0)

C:\Users\DELL\AppData\Local\Temp\ipykernel_2656\4276739279.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["fee"] = df["fee"].fillna(df["fee"].median())
C:\Users\DELL\AppData\Local\Temp\ipykernel_2656\4276739279.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["reward"] = df["reward"].fillna(0)


6. Date Columns

In [142]:
date_columns = [
    "created_at",
    "modified_at",
    "last_date_to_apply",
    "apply_date",
    "accept_reject_date",
    "completion_date",
    "reward_awarded_date",
    "withdraw_date",
    "dropped_out_date",
    "not_started_date"
]

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

C:\Users\DELL\AppData\Local\Temp\ipykernel_2656\869465981.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_datetime(df[col], errors="coerce")
C:\Users\DELL\AppData\Local\Temp\ipykernel_2656\869465981.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = pd.to_datetime(df[col], errors="coerce")
C:\Users\DELL\AppData\Local\Temp\ipykernel_2656\869465981.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_

7. Marketing Columns

In [143]:
marketing_columns = [
    "from_where_did_you_hear_about_us",
    "from_which_medium_did_you_hear_about_the_internship",
    "how_did_you_hear_about_this_opportunity"
]

for col in marketing_columns:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

C:\Users\DELL\AppData\Local\Temp\ipykernel_2656\1566644676.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].fillna("Unknown")


8. Team Columns

In [144]:
team_columns = [
    "team_name",
    "team_code",
    "team_creator"
]

for col in team_columns:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip())

C:\Users\DELL\AppData\Local\Temp\ipykernel_2656\1797969475.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = (


In [145]:
df.isnull().sum().sort_values(ascending=False)

please_identify_your_student_status                    14794
team_creator                                           14787
dropouttransaction                                     14747
notstartedtransaction                                  14716
transaction_id                                         14602
                                                       ...  
payment_status                                             0
status                                                     0
fee                                                        0
from_which_medium_did_you_hear_about_the_internship        0
how_did_you_hear_about_this_opportunity                    0
Length: 65, dtype: int64

Validation after cleaning

In [146]:
print("Total Rows:", df.shape[0])
print("Total Columns:", df.shape[1])
print("Duplicate Rows:", df.duplicated().sum())

remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

print("\nColumns with Remaining Missing Values:")
print(remaining_missing)

Total Rows: 14799
Total Columns: 65
Duplicate Rows: 0

Columns with Remaining Missing Values:
badge                                    401
careeraddon                            12912
opportunity_category                       4
code                                       5
cohort                                    23
opportunity_created_at                     5
current_editor                         13313
dropouttransaction                     14747
duration                                   4
eligibility                              353
opportunity_fee                            4
image_link                                57
last_date_to_apply                         5
opportunity_modified_at                    4
name                                       5
notstartedtransaction                  14716
panellist                              11802
role                                    8768
accept_reject_date                      8051
amount_to_be_paid                       5753
transa

data types check

In [147]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14799 entries, 0 to 14798
Data columns (total 65 columns):
 #   Column                                               Non-Null Count  Dtype              
---  ------                                               --------------  -----              
 0   opportunity_pk                                       14799 non-null  object             
 1   opportunity_id                                       14799 non-null  object             
 2   badge                                                14398 non-null  object             
 3   careeraddon                                          1887 non-null   object             
 4   opportunity_category                                 14795 non-null  object             
 5   code                                                 14794 non-null  object             
 6   cohort                                               14776 non-null  object             
 7   opportunity_created_at                  

In [148]:
df.amount_to_be_paid.value_counts()

amount_to_be_paid
0.0      6638
0         692
1.0       633
100.0     310
10.0      185
         ... 
86.0        1
45.0        1
36.0        1
80          1
8           1
Name: count, Length: 118, dtype: int64

Changing the datatype

In [149]:
df['amount_to_be_paid']= pd.to_numeric(df['amount_to_be_paid'], errors='coerce')

C:\Users\DELL\AppData\Local\Temp\ipykernel_2656\1913785580.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['amount_to_be_paid']= pd.to_numeric(df['amount_to_be_paid'], errors='coerce')


In [150]:
df['opportunity_fee']=pd.to_numeric(df['opportunity_fee'], errors='coerce')

C:\Users\DELL\AppData\Local\Temp\ipykernel_2656\2830348205.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['opportunity_fee']=pd.to_numeric(df['opportunity_fee'], errors='coerce')


In [151]:
df['duration']= pd.to_numeric(df['duration'], errors='coerce')

C:\Users\DELL\AppData\Local\Temp\ipykernel_2656\998528511.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['duration']= pd.to_numeric(df['duration'], errors='coerce')


checking the datatype

In [152]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14799 entries, 0 to 14798
Data columns (total 65 columns):
 #   Column                                               Non-Null Count  Dtype              
---  ------                                               --------------  -----              
 0   opportunity_pk                                       14799 non-null  object             
 1   opportunity_id                                       14799 non-null  object             
 2   badge                                                14398 non-null  object             
 3   careeraddon                                          1887 non-null   object             
 4   opportunity_category                                 14795 non-null  object             
 5   code                                                 14794 non-null  object             
 6   cohort                                               14776 non-null  object             
 7   opportunity_created_at                  

empty strings

In [153]:
(df == "").sum().sort_values(ascending=False)

opportunity_pk                             0
opportunity_id                             0
badge                                      0
careeraddon                                0
opportunity_category                       0
                                          ..
external_reference_url                     0
work_item_sk                               0
how_did_you_hear_about_this_opportunity    0
dropped_out_date                           0
terms_conditions_accepted                  0
Length: 65, dtype: Int64

again connect with POstgreSQL

In [157]:
df.to_sql(
    name="cleaned_data",
    con=engine,
    if_exists="replace",
    index=False
)

212